# DAT535 Project – Group 24: Norgespris and electricity use
## Notebook 02: Silver layer

**Input:** the `bronze` schema in PostgreSQL (raw text tables, loaded by notebook 01 / stage `postgres`).
**Output:** the `silver` schema in PostgreSQL – clean, typed, de-duplicated tables, plus a rejects table and quality metrics.

```text
bronze.* (PostgreSQL, all text) ──(Spark: read over JDBC, clean, standardise, enrich)──▶ silver.* (PostgreSQL, typed)
```

This notebook implements section 4 of our *Data Engineering Project Definition Guide (v3)*. The guide asks for the most detail here, so
section 1 answers the guide's questions one by one, and the code in sections 2–5 follows those answers.

**How it fits the repo**
* Same pattern as notebook 01: the code cells marked as pipeline code are also in **`silver_pipeline.py`** (generated from this notebook),
  so GitHub Actions runs exactly this code: `python run_pipeline.py silver` (and `all` = `bronze` → `postgres` → `silver`).
* It reuses the environment from notebook 01: `.env`, `NORGESPRIS_ENV` (dev/prod → database `norgespris_dev`/`norgespris_prod`),
  and `get_spark()` + the PostgreSQL settings from `bronze_to_postgres.py`.
* Tests: `tests/test_silver.py` runs the cleaning functions on tiny DataFrames (no database needed).

---
# 1. Silver design (answers to section 4 of the definition guide)

### 1.1 Data understanding – what one Bronze record is

| Bronze table | One record is | Essential fields | Optional / unreliable |
|---|---|---|---|
| `consumption_per_group_municipality_hour` | kWh for **one municipality, one hour, one consumption group** | `START_TIME`, `MUNICIPALITY_NUMBER`, `CONSUMPTION_GROUP`, `QUANTITY_KWH`, `LAST_UPDATE_TIME` | `NAME` (broken letters seen), `METERING_POINT_COUNT`; municipality `0` = unknown |
| `population_07459` | one **municipality** with one column **per year** (wide) | region text (`K-3101 Halden`), year columns | non-municipality rows (`K-23 …`, `K-Rest …`) |
| `frost_stations` | one **weather station** | `id`, `municipalityId` | `geometry` (JSON text), `countyId`; stations at sea/abroad have no municipality; codes may be pre-2024 |
| `frost_observations` | one **station × day**, with a JSON list of observations | `sourceId`, `referenceTime`, `observations` | several daily-mean series per day (`timeOffset`) |
| `norgespris_count_per_municipality_consumption_group` | number of metering points **with Norgespris** and **in total** for **one day × municipality × consumption group** | `DATE`, `MUNICIPALITY_NUMBER`, `CONSUMPTION_GROUP`, `COUNT_NORWAYPR…`, `COUNT` | `MUNICIPALITY` (name); group names differ from the consumption data (`Household`, `Cabins and holiday properties`) |
| `norgespris_consumption_per_group_eac_mba` | kWh with/without Norgespris for **one day × price area × group × consumption band** | `START_TIME`, `PRICE_AREA`, `CONSUMPTION_GROUP`, kWh columns | band limits empty for some rows |

Every Bronze row also has `_source_file`, `_landing_run_id`, `_loaded_at` (lineage from notebook 01).

### 1.2 Data model – grain, keys, relationships

| Silver table | Grain (= one row per …) | Key |
|---|---|---|
| `silver.municipality` | municipality (today's structure, from SSB) | `municipality_no` |
| `silver.population_yearly` | municipality × year | `municipality_no, year` |
| `silver.consumption_daily` | municipality × **local** date × consumption group | `municipality_no, date, consumption_group` |
| `silver.weather_station` | weather station | `station_id` |
| `silver.temperature_station_daily` | station × date | `station_id, date` |
| `silver.temperature_municipality_daily` | municipality × date | `municipality_no, date` |
| `silver.norgespris_count_daily` | municipality × date × consumption group | `municipality_no, date, consumption_group` |
| `silver.norgespris_price_area_daily` | date × price area × group × consumption band | `date, price_area, consumption_group, eac_from_kwh, eac_to_kwh` |
| `silver.rejects` | one rejected Bronze record | – (rule id + the record as JSON) |
| `silver.quality_metrics` | one metric of one run (history, appended) | `run_id, table_name, metric` |

**`municipality_no` (4-character text, e.g. `0301`) is the common key** for consumption, population and temperature; `silver.municipality` is the reference.
**Why daily and not hourly in Silver:** our question is per day; hourly Elhub data is ~15 million rows per year. Silver cleans it **per hour**
(types, duplicates, rejects) and then sums it to the local day (~0.5 million rows per year) – much faster to write to and read from PostgreSQL.
The hourly detail stays in Bronze if we ever need it.
**Period:** Silver keeps only the months of the **same period as the Frost download** – `NORGESPRIS_<ENV>_START_MONTH` / `_END_MONTH` in `.env`
(prod: `2024-01` → last complete month). One setting for the whole pipeline means every Silver table covers exactly the same days, and
Bronze can hold more (Elhub's export always has ~3 years) without Silver changing. Population: the years of that period.
**History:** Silver is rebuilt from Bronze on every run (overwrite). Reference data (municipalities, stations) is the current version – no history (no SCD) – on purpose, to keep it simple.

### 1.3 Cleaning rules

Every rule has an **id**; records that break a rule go to `silver.rejects` with that id and the original record – they are not silently dropped.

| Rule | Dataset | A record is rejected when … |
|---|---|---|
| `CONS-01` | consumption | `START_TIME` missing or not a timestamp |
| `CONS-02` | consumption | `QUANTITY_KWH` missing or not a number |
| `CONS-03` | consumption | kWh is negative |
| `CONS-04` | consumption | municipality number missing |
| `CONS-05` | consumption | consumption group missing |
| `SSB-01` | population | region has no 4-digit municipality code (e.g. `K-23 Kontinentalsokkelen`, `K-Rest …`) |
| `SSB-02` | population | population missing, not a number, or ≤ 0 |
| `ST-01` | stations | station id missing |
| `TEMP-01` | temperature | date missing / not a date |
| `TEMP-02` | temperature | value missing, or outside −60 … +45 °C (physically impossible in Norway) |
| `NP-01` | Norgespris | date missing / not a date |
| `NP-02` | Norgespris | price area missing |
| `NP-03` | Norgespris | a kWh value is negative |
| `NPC-01` | Norgespris count | `DATE` missing / not a date |
| `NPC-02` | Norgespris count | municipality number missing |
| `NPC-03` | Norgespris count | a count is missing or negative, or more meters with Norgespris than in total |

**Duplicates:** for every key we keep **the newest version** (latest `LAST_UPDATE_TIME`, then latest landing run). This also handles revised Elhub data and overlapping exports.
**Missing values that are not errors** are kept and *flagged* instead: municipality `0`/unknown codes → `known_municipality = false`; incomplete days → `complete_day = false`; stations without a municipality → `municipality_no` empty.

### 1.4 Standardisation
* **Schemas are defined before the code** (`SILVER_SCHEMAS`, section 2.3); every table is cast to its schema before it is written – a missing column stops the run.
* **Municipality numbers:** always 4 characters (`lpad`): `301` → `0301`, `0` → `0000`.
* **Dates = Norwegian local dates everywhere**, so consumption and temperature are joined on the same day:
  * Elhub: the first 10 characters of `START_TIME`, which Elhub already writes in local time (`+01:00/+02:00`). We do *not* convert to UTC first – that would shift every day by 1–2 hours.
  * Frost: `referenceTime` is UTC (`…Z`). We convert it to Norwegian time with the time zone **`Europe/Oslo`** (= CET, +01:00, in winter and CEST, +02:00, in summer – the zone name handles the switch) and then take the date: `convert_timezone('UTC', 'Europe/Oslo', …)`.
* **Numbers:** kWh as `DECIMAL(18,3)`, temperature as `DECIMAL(5,2)`, counts as `INT`. Text → lower case where it is a category (`household`, `cabin`, …).
* **Consumption groups** use the same names in every Silver table: the Norgespris count file says `Household` / `Cabins and holiday properties`,
  the consumption file `household` / `cabin` → both become `household` / `cabin` (`GROUP_NAMES` in 4.5), so Gold can join them directly.
* **Names** come from SSB only (Elhub's names can have broken letters) – we join on numbers, never on names, except the explicit name fallback below.

### 1.5 Enrichment
* **Municipality matching** (`match_municipality`): a code that exists in SSB's list is used as it is; a code that does not (e.g. a pre-2024 code like Drammen `3005`) is matched **by name** if that name is unique in SSB's list (Drammen → `3301`). This repairs the 2024 renumbering without a separate mapping table.
* **Temperature per municipality:** average of all stations in the municipality; if it has none, the **county average** (`county_no` = first two digits of `municipality_no`). The column `method` says which (`stations` / `county_average`).
* `known_municipality` on consumption and Norgespris-count rows (is the code in SSB's list?).
* `norgespris_share` = metering points with Norgespris ÷ all metering points (per municipality, day and group) – the share Gold needs.

### 1.6 Quality evidence
`silver.quality_metrics` gets one row per metric per run (appended, never overwritten), e.g. rows read from Bronze, rejected, duplicates removed, rows written, kWh in unknown municipalities, municipalities with own stations vs county average.
Section 5.2 compares the newest run with the previous one – that is the "before and after" evidence the guide asks for.

### 1.7 Implementation in Spark
* **DataFrame API** with built-in functions (`try_cast`, `lpad`, `from_json`, `explode`, window `row_number`) for the cleaning – no Python UDFs (they are slow, Spark cannot optimise them).
* **Spark SQL** for the temperature enrichment (section 4.3) – a join and two aggregations read best as SQL; Gold will use Spark SQL as well.
* `try_cast` instead of `cast`: in Spark 4 (ANSI mode) a plain `cast('abc' AS DECIMAL)` stops the whole job; `try_cast` returns NULL, which our reject rules then catch.

### 1.8 Performance – where it matters, and what we do
| Where | Risk | What we do |
|---|---|---|
| Reading the big Elhub table over JDBC | one connection, one partition, all rows in the driver's memory | `fetchsize=10000` (stream rows) and **one partition per month** (`predicates`), so Spark reads in parallel |
| Using the same input several times (rejects, metrics, clean data) | each use would read PostgreSQL again | `persist()` the Bronze input once, `unpersist()` after |
| Duplicate removal + daily sums | shuffle (`groupBy`, window) | done once on the hourly data; `spark.sql.shuffle.partitions=8` and adaptive query execution (from `get_spark()`) |
| Joins with municipalities / stations | shuffle of the big side | small tables are **broadcast** (`F.broadcast`) |
| Skew | – | little expected: data is spread evenly over municipalities and hours |

Tuning is kept to these few steps; section 5.3 shows the evidence (partitions, plan, time) instead of tuning "by assumption".

### 1.9 Lifecycle and scalability
* **Tests:** `tests/test_silver.py` (pytest) runs every cleaning function on tiny DataFrames – part of every workflow run.
* **Review/versioning:** changes go through a pull request `dev_branch → main`, like the rest of the repo.
* **Deploy/rerun/monitor:** stage `silver` in `run_pipeline.py`; a rerun simply rebuilds Silver from Bronze (idempotent). Run status in GitHub Actions, rejects and metrics in PostgreSQL.
* **Larger volume:** more months → more JDBC partitions (one per month), nothing else changes. **Late data:** handled by "newest version per key". **Schema change:** a missing Bronze column makes the run fail with a clear error (no half-written Silver).

---
# 2. Setup

### 2.1 Imports and shared settings
The notebook runs in `notebooks/`, so we add the repo root to Python's search path first, to be able to import the shared code from notebook 01.

In [1]:
import os, sys                                                  # paths
sys.path.insert(0, os.path.abspath(".."))                       # repo root -> import bronze_pipeline, bronze_to_postgres

In [2]:
import logging                                                  # log messages (also visible in GitHub Actions)
import re                                                       # find year columns and municipality codes in SSB text
import time                                                     # elapsed time
from datetime import date, datetime, timedelta, timezone       # run id, timestamps (UTC) and the Silver period

from pyspark import StorageLevel                                # persist big inputs in memory, spill to disk if needed
from pyspark.sql import functions as F                          # built-in column functions
from pyspark.sql.window import Window                           # "newest version per key" (row_number over a window)

from bronze_pipeline import ENV, month_range, last_complete_month, next_month  # dev/prod + the period from .env
from bronze_to_postgres import get_spark, PG_URL, PG_PROPERTIES, PG_PASSWORD  # same Spark + database settings as Bronze

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger("silver")                            # own logger name in the run log

spark = None                                                    # the SparkSession, set in run_silver() / the notebook
JDBC_READ = dict(PG_PROPERTIES, fetchsize="10000")              # stream 10 000 rows at a time (default: ALL rows into memory)

2026-11-13 01:50:14,092 - bronze - INFO - Loaded ['FROST_CLIENT_ID', 'FROST_CLIENT_SECRET', 'NORGESPRIS_DEV_START_MONTH', 'NORGESPRIS_DEV_END_MONTH', 'NORGESPRIS_PROD_START_MONTH', 'NORGESPRIS_PROD_END_MONTH', 'NORGESPRIS_LANDING_ROOT', 'NORGESPRIS_PG_HOST', 'NORGESPRIS_PG_PORT', 'NORGESPRIS_PG_USER', 'NORGESPRIS_PG_PASSWORD'] from /Users/x/UiS/DAT535 Data Engineering/dat535-group24/.env


### 2.2 Reading from and writing to PostgreSQL
* `pg_query(sql)` – run a `SELECT` inside PostgreSQL and get the answer as a DataFrame (used for small things: table lists, counts, months).
* `read_bronze(table, split_on)` – read a Bronze table. With `split_on` (a column that starts with `YYYY-MM`) Spark reads **one month per partition**
  in parallel. The last predicate catches every row that does not start with a valid month, so **every row is read exactly once**.
* `write_silver(df, table)` – replace a Silver table (`overwrite`) in batches of 10 000 rows, then count the rows **in PostgreSQL**.

In [3]:
def pg_query(sql):
    """Run a SELECT inside PostgreSQL and return the result as a Spark DataFrame."""
    return spark.read.jdbc(PG_URL, f"({sql}) AS q", properties=JDBC_READ)


def bronze_tables():
    """Names of the tables that exist in the bronze schema."""
    rows = pg_query("SELECT table_name FROM information_schema.tables WHERE table_schema = 'bronze'").collect()
    return {r["table_name"] for r in rows}


def read_bronze(table, split_on=None):
    """Read bronze.<table>; with split_on: one partition per month (YYYY-MM prefix of that column)."""
    if split_on is None:                                        # small table -> one partition is fine
        return spark.read.jdbc(PG_URL, f"bronze.{table}", properties=JDBC_READ)
    col = f'"{split_on}"'                                       # quoted: Postgres column names are upper case here
    months = [r["m"] for r in pg_query(f"SELECT DISTINCT left({col}, 7) AS m FROM bronze.{table}").collect()]
    months = sorted(m for m in months if m and re.fullmatch(r"\d{4}-\d{2}", m))  # only clean YYYY-MM values in SQL
    predicates = [f"left({col}, 7) = '{m}'" for m in months]    # one WHERE clause = one Spark partition
    predicates.append(f"{col} IS NULL OR left({col}, 7) !~ '^[0-9]{{4}}-[0-9]{{2}}$'")  # everything else (bad dates)
    return spark.read.jdbc(PG_URL, f"bronze.{table}", predicates=predicates, properties=JDBC_READ)


def write_silver(df, table):
    """Replace silver.<table> with df and return the number of rows now in PostgreSQL."""
    (df.write
       .option("batchsize", 10000)                              # rows per INSERT batch
       .jdbc(PG_URL, f"silver.{table}", mode="overwrite", properties=PG_PROPERTIES))
    rows = pg_query(f"SELECT COUNT(*) AS n FROM silver.{table}").first()["n"]  # count inside the database
    logger.info(f"  silver.{table}: {rows:,} rows")
    return rows

### 2.3 Silver schemas – defined before the code
One list of `(column, type)` per table. `conform()` selects exactly these columns, in this order, cast to these types.
If a column is missing, Spark stops with an error before anything is written – that is our schema check.
`KEYS` documents the grain of each table and is used for the "duplicate keys" quality check.
`silver_period()` turns the months in `.env` into a first and last day (e.g. `2024-01` → `2024-01-01`, last complete month → its last day);
`in_period()` keeps only the rows in between. The filter is applied in `run_silver()` (5.1), so the cleaning functions in section 4 stay simple and testable.

In [4]:
SILVER_SCHEMAS = {
    "municipality": [("municipality_no", "string"), ("municipality_name", "string"), ("county_no", "string")],
    "population_yearly": [("municipality_no", "string"), ("year", "int"), ("population", "int")],
    "consumption_daily": [("municipality_no", "string"), ("date", "date"), ("consumption_group", "string"),
                          ("kwh", "decimal(18,3)"), ("metering_points", "int"), ("n_hours", "int"),
                          ("complete_day", "boolean"), ("known_municipality", "boolean"),
                          ("last_update_time", "timestamp")],
    "weather_station": [("station_id", "string"), ("station_name", "string"), ("municipality_no", "string"),
                        ("municipality_match", "string"), ("county_no", "string"),
                        ("latitude", "double"), ("longitude", "double")],
    "temperature_station_daily": [("station_id", "string"), ("date", "date"), ("mean_temp_c", "decimal(5,2)"),
                                  ("time_offset", "string")],
    "temperature_municipality_daily": [("municipality_no", "string"), ("date", "date"),
                                       ("mean_temp_c", "decimal(5,2)"), ("n_stations", "int"), ("method", "string")],
    "norgespris_price_area_daily": [("date", "date"), ("price_area", "string"), ("consumption_group", "string"),
                                    ("eac_from_kwh", "decimal(18,3)"), ("eac_to_kwh", "decimal(18,3)"),
                                    ("kwh_norgespris", "decimal(18,3)"), ("kwh_not_norgespris", "decimal(18,3)"),
                                    ("kwh_total", "decimal(18,3)"), ("metering_points_norgespris", "int"),
                                    ("metering_points_not_norgespris", "int"), ("metering_points_total", "int")],
    "norgespris_count_daily": [("municipality_no", "string"), ("date", "date"), ("consumption_group", "string"),
                               ("metering_points_norgespris", "int"), ("metering_points_total", "int"),
                               ("norgespris_share", "decimal(6,4)"), ("known_municipality", "boolean")],
}

KEYS = {                                                        # the grain of every table (= what must be unique)
    "municipality": ["municipality_no"],
    "population_yearly": ["municipality_no", "year"],
    "consumption_daily": ["municipality_no", "date", "consumption_group"],
    "weather_station": ["station_id"],
    "temperature_station_daily": ["station_id", "date"],
    "temperature_municipality_daily": ["municipality_no", "date"],
    "norgespris_price_area_daily": ["date", "price_area", "consumption_group", "eac_from_kwh", "eac_to_kwh"],
    "norgespris_count_daily": ["municipality_no", "date", "consumption_group"],
}


def conform(df, table):
    """Exactly the columns of SILVER_SCHEMAS[table], in order, with the defined types."""
    return df.select(*[F.col(name).cast(dtype).alias(name) for name, dtype in SILVER_SCHEMAS[table]])


def silver_period():
    """(first day, last day) of the period in .env - the same months the Frost download uses."""
    start, end = month_range()                                  # e.g. ('2024-01', None) in prod
    end = end or last_complete_month()                          # None = up to the last complete month
    first_day = date.fromisoformat(f"{start}-01")               # 2024-01 -> 2024-01-01
    last_day = date.fromisoformat(f"{next_month(end)}-01") - timedelta(days=1)  # 2026-10 -> 2026-10-31
    return first_day, last_day


def in_period(df, first_day, last_day):
    """Only the rows whose date lies in the period (both days included)."""
    return df.filter(F.col("date").between(first_day, last_day))

### 2.4 Shared cleaning helpers
* `try_cast(column, type)` – like `cast`, but returns NULL instead of stopping the job when a value cannot be converted (Spark 4 runs in ANSI mode).
* `col_or_null(df, name)` – the column if it exists, otherwise NULL (Frost leaves out fields that are empty for all stations).
* `split_rejects(df, rules, dataset)` – gives every record the id of the **first** rule it breaks; returns the good records and the rejected ones (as JSON text, for `silver.rejects`).
* `newest_per_key(df, keys, order_cols)` – keeps one row per key: the first one when sorted by `order_cols` (newest first).
* `match_municipality(...)` – the code-then-unique-name matching from 1.5. Names are compared with `name_key`: upper case and only the part
  before ` - `, because SSB writes Sámi/Kven names too (`Oslo - Oslove`, `Tromsø - Romsa - Tromssa`), Frost only the Norwegian one.

In [5]:
def try_cast(column, dtype):
    """Cast a column; NULL instead of an error when the value cannot be converted."""
    return F.expr(f"try_cast(`{column}` AS {dtype})")


def col_or_null(df, name, dtype="string"):
    """df[name] if the column exists, otherwise a NULL column of the given type."""
    return F.col(f"`{name}`") if name in df.columns else F.lit(None).cast(dtype)


def split_rejects(df, rules, dataset):
    """rules = [(rule_id, condition that is TRUE for a bad record), ...] -> (good_df, rejects_df)."""
    reason = F.lit(None).cast("string")
    for rule_id, is_bad in reversed(rules):                     # build "first rule that fails" from the back
        reason = F.when(F.coalesce(is_bad, F.lit(False)), F.lit(rule_id)).otherwise(reason)
    flagged = df.withColumn("_reject_rule", reason)
    good = flagged.filter(F.col("_reject_rule").isNull()).drop("_reject_rule")
    bad = flagged.filter(F.col("_reject_rule").isNotNull()).select(
        F.lit(dataset).alias("dataset"),                        # which Silver table it was meant for
        F.col("_reject_rule").alias("rule_id"),                 # which rule it broke
        F.to_json(F.struct(*[F.col(f"`{c}`") for c in df.columns])).alias("record"))  # the record itself
    return good, bad


def newest_per_key(df, keys, order_cols):
    """One row per key: the first row when sorted by order_cols (use F.desc(...) for newest first)."""
    w = Window.partitionBy(*keys).orderBy(*order_cols)
    return df.withColumn("_rank", F.row_number().over(w)).filter("_rank = 1").drop("_rank")


def name_key(column):
    """Comparable name: upper case, only the part before ' - ' (SSB adds Sami/Kven names: 'Tromsø - Romsa - Tromssa')."""
    return F.upper(F.trim(F.regexp_replace(column, r"\s+-\s+.*$", "")))


def match_municipality(df, code_col, name_col, municipalities):
    """Add municipality_no + municipality_match ('code' / 'name' / None) using SSB's municipality list."""
    ref = municipalities.select(F.col("municipality_no").alias("_ref_no"),
                                name_key(F.col("municipality_name")).alias("_ref_name"))
    unique_names = (ref.groupBy("_ref_name").agg(F.count("*").alias("_n"), F.first("_ref_no").alias("_name_no"))
                       .filter("_n = 1").drop("_n"))            # names that point to exactly one municipality
    out = (df.join(F.broadcast(ref.select("_ref_no")), F.col(code_col) == F.col("_ref_no"), "left")
             .join(F.broadcast(unique_names), name_key(F.col(name_col)) == F.col("_ref_name"), "left"))
    return (out.withColumn("municipality_match",
                           F.when(F.col("_ref_no").isNotNull(), "code")
                            .when(F.col("_name_no").isNotNull(), "name"))
               .withColumn("municipality_no", F.coalesce("_ref_no", "_name_no", F.col(code_col)))
               .drop("_ref_no", "_ref_name", "_name_no"))

---
# 3. Start Spark and look at the input

`get_spark()` comes from `bronze_to_postgres.py`: same settings and the PostgreSQL JDBC driver.
⚠️ If you ran notebook 01 in the same Jupyter session, the session is reused – that is fine.

In [6]:
spark = get_spark()                                             # Spark with the PostgreSQL driver
print("Silver period:", silver_period())                        # first and last day from .env
print("Spark", spark.version, "| env:", ENV, "| database:", PG_URL)
print("Bronze tables:", sorted(bronze_tables()))                # what notebook 01 loaded

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/13 01:50:16 WARN Utils: Your hostname, X-sin-MacBook-Pro-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.4.120 instead (on interface en0)
26/11/13 01:50:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/x/UiS/DAT535%20Data%20Engineering/spark-env/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/x/.ivy2.5.2/cache
The jars for the packages stored in: /Users/x/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-35a8b2a5-5fee-424f-84dc-b1cf9d1d8cd6;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.4 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 124ms :: artifacts dl 5ms
	:: modules in use:
	org.

Silver period: (datetime.date(2024, 1, 1), datetime.date(2026, 11, 30))
Spark 4.3.0 | env: dev | database: jdbc:postgresql://localhost:5432/norgespris_dev
Bronze tables: ['_load_log', 'consumption_per_group_municipality_hour', 'frost_observations', 'frost_stations', 'frost_timeseries', 'manifest', 'norgespris_consumption_per_group_eac_mba', 'norgespris_count_per_municipality_consumption_group', 'population_07459']


---
# 4. The Silver tables

Each step is a **function that takes DataFrames and returns DataFrames** (no reading or writing inside). That makes them easy to test
(`tests/test_silver.py`) and to reuse in `run_silver()` (section 5).

### 4.1 Municipalities and population (SSB)
The SSB table is **wide**. Two layouts exist:

| | region column | year columns | region text |
|---|---|---|---|
| manual export | `Kommune` | `2023`, `2024`, … | `K-3101 Halden` |
| API (`usecodesandtexts`) | `Region - region` | `Personer1 - Personer 2023 - 2023`, … | `K-3101 - K-3101 Halden` |

We find the region column by name (contains *region* or *kommune*) and the year columns by the 4-digit year in their name, so both work
(also if Bronze holds both kinds of files: `coalesce` takes whichever column has a value). The years become rows (`explode` of an array of
`(year, value)` pairs = unpivot). Code = the first 4-digit number; **name = the text after the last code** (`K-3101 - K-3101 Halden` → `Halden`;
names never contain digits, Sámi names stay: `Oslo - Oslove`).
`silver.municipality` is built from the newest year of each municipality; `county_no` = the first two digits of the code.
The name is only there if SSB delivers it: the API writes **only codes** in CSV unless the request asks for `usecodesandtexts`
(fixed in `SSB_URL`, notebook 01). The metric `municipalities_without_name` (must be 0) shows if that ever breaks again.

In [7]:
def clean_population(bronze):
    """bronze.population_07459 (wide) -> (population_yearly, municipality, rejects)."""
    region_cols = ([c for c in bronze.columns if re.search(r"region|kommune", c, re.IGNORECASE)]
                   or [bronze.columns[0]])                      # 'Kommune' (manual export) / 'Region - region' (API)
    year_of = {c: re.search(r"(19|20)\d{2}", c).group(0) for c in bronze.columns
               if c not in region_cols and not c.startswith("_") and re.search(r"(19|20)\d{2}", c)}  # column -> year
    years = sorted(set(year_of.values()))                       # '2023', '2024', ...
    pairs = F.array(*[F.struct(F.lit(int(y)).alias("year"),     # one (year, value) pair per year ...
                               F.coalesce(*[F.col(f"`{c}`") for c, cy in year_of.items() if cy == y]).alias("value"))
                      for y in years])                          # ... value from whichever column has that year
    region = F.coalesce(*[F.col(f"`{c}`") for c in region_cols])  # the region text, whichever format
    long = (bronze.select(region.alias("region"), "_landing_run_id", F.explode(pairs).alias("p"))
                  .select("region", "_landing_run_id", "p.year", "p.value"))  # one row per municipality and year
    df = (long.withColumn("municipality_no", F.regexp_extract("region", r"(\d{4})", 1))   # 'K-3101 - K-3101 Halden' -> '3101'
              .withColumn("municipality_name", F.trim(F.regexp_replace("region", r"^.*\d{4}\s*", "")))  # after the LAST code -> 'Halden'
              .withColumn("population", try_cast("value", "int")))
    good, rejects = split_rejects(df, [
        ("SSB-01", F.col("municipality_no") == ""),             # K-23 Kontinentalsokkelen, K-Rest ...
        ("SSB-02", F.col("population").isNull() | (F.col("population") <= 0)),
    ], "population_yearly")
    good = newest_per_key(good, ["municipality_no", "year"], [F.desc("_landing_run_id")])  # same year twice -> newest file
    municipality = (newest_per_key(good, ["municipality_no"], [F.desc("year")])  # name from the newest year
                    .withColumn("county_no", F.substring("municipality_no", 1, 2)))
    return conform(good, "population_yearly"), conform(municipality, "municipality"), rejects

### 4.2 Electricity consumption (Elhub) – hourly → daily
1. Types: `try_cast` of time, kWh and counts; municipality number padded to 4 characters; group in lower case.
2. **Local date** = first 10 characters of `START_TIME` (it is written in Norwegian local time).
3. Reject rules `CONS-01 … CONS-05`.
4. **Duplicates:** one row per hour, municipality and group – the newest `LAST_UPDATE_TIME` wins (revised data, overlapping exports, `301` vs `0301`).
5. `known_municipality`: is the code (or, for old codes, the name) in SSB's list? Municipality `0` stays in Silver (its kWh are real) but is marked.
6. **Sum per local day.** `n_hours` should be 24, or 23/25 on the days the clock changes (end of March / October) – otherwise `complete_day = false`.

In [8]:
def clean_consumption(bronze, municipalities):
    """bronze.consumption_per_group_municipality_hour -> (consumption_daily, rejects, number of duplicates removed)."""
    df = bronze.select(
        F.col("START_TIME").alias("start_time_text"),
        try_cast("START_TIME", "timestamp").alias("start_time"),              # instant in time (offset understood)
        F.expr("try_cast(substring(START_TIME, 1, 10) AS DATE)").alias("date"),  # LOCAL date, as written by Elhub
        F.lpad(F.trim("MUNICIPALITY_NUMBER"), 4, "0").alias("municipality_code"),  # '301' -> '0301', '0' -> '0000'
        F.col("NAME").alias("name"),
        F.lower(F.trim("CONSUMPTION_GROUP")).alias("consumption_group"),
        try_cast("QUANTITY_KWH", "decimal(18,3)").alias("kwh"),
        try_cast("METERING_POINT_COUNT", "int").alias("metering_points"),
        try_cast("LAST_UPDATE_TIME", "timestamp").alias("last_update_time"),
        F.col("_landing_run_id"))
    good, rejects = split_rejects(df, [
        ("CONS-01", F.col("start_time").isNull() | F.col("date").isNull()),
        ("CONS-02", F.col("kwh").isNull()),
        ("CONS-03", F.col("kwh") < 0),
        ("CONS-04", F.col("municipality_code").isNull() | (F.col("municipality_code") == "")),
        ("CONS-05", F.col("consumption_group").isNull() | (F.col("consumption_group") == "")),
    ], "consumption_daily")
    good = match_municipality(good, "municipality_code", "name", municipalities)  # old codes -> today's code
    deduped = newest_per_key(good, ["start_time", "municipality_no", "consumption_group"],
                             [F.desc_nulls_last("last_update_time"), F.desc("_landing_run_id")])
    daily = (deduped.groupBy("municipality_no", "date", "consumption_group")
                    .agg(F.sum("kwh").alias("kwh"),                          # kWh per local day
                         F.max("metering_points").alias("metering_points"),  # metering points that day
                         F.count("*").alias("n_hours"),                      # hours with data
                         F.max("last_update_time").alias("last_update_time"),
                         F.max(F.col("municipality_match").isNotNull()).alias("known_municipality"))
                    .withColumn("complete_day", F.col("n_hours").between(23, 25)))  # 23/25 = daylight saving days
    return conform(daily, "consumption_daily"), rejects, deduped

### 4.3 Weather stations and temperature (Frost)
**Stations:** id, name, coordinates (from the `geometry` JSON text) and municipality – matched with `match_municipality` (old codes are repaired by name).
**Temperature per station and day:** the observation list (JSON text in Bronze) is parsed with `from_json` and exploded; we keep only
`mean(air_temperature P1D)`, reject impossible values, and keep **one value per station and day**: the `PT0H` series (midnight–midnight UTC)
if it exists, otherwise the lowest offset; newest landing run first.
**Date in Norwegian time:** `referenceTime` (UTC) → `Europe/Oslo` (CET/CEST) → date. `try_cast(… AS TIMESTAMP_NTZ)` reads the first 19 characters
as a plain date-time without time zone, and `convert_timezone` moves it from UTC to Oslo time – explicit, so the result does not depend on the time zone of the
computer Spark runs on. Note: a `PT0H` daily mean still *covers* 00–24 UTC (= 01–01 / 02–02 Norwegian time); the conversion gives it the Norwegian date label.
**Temperature per municipality and day – in Spark SQL:** every municipality × every day; the average of its own stations, otherwise the county average.

In [9]:
OBS_SCHEMA = "array<struct<elementId:string,value:double,timeOffset:string,qualityCode:int>>"  # Frost observation list


def clean_stations(bronze, municipalities):
    """bronze.frost_stations -> (weather_station, rejects)."""
    point = F.from_json(col_or_null(bronze, "geometry"), "struct<coordinates:array<double>>")  # {"coordinates":[lon,lat]}
    df = bronze.select(
        F.trim(col_or_null(bronze, "id")).alias("station_id"),
        col_or_null(bronze, "name").alias("station_name"),
        F.lpad(F.trim(col_or_null(bronze, "municipalityId")), 4, "0").alias("municipality_code"),  # 301 -> 0301
        col_or_null(bronze, "municipality").alias("frost_municipality_name"),
        point["coordinates"][1].alias("latitude"),
        point["coordinates"][0].alias("longitude"),
        F.col("_landing_run_id"))
    good, rejects = split_rejects(df, [("ST-01", F.col("station_id").isNull() | (F.col("station_id") == ""))],
                                  "weather_station")
    good = newest_per_key(good, ["station_id"], [F.desc("_landing_run_id")])
    good = match_municipality(good, "municipality_code", "frost_municipality_name", municipalities)
    good = (good.withColumn("municipality_no", F.when(F.col("municipality_match").isNotNull(), F.col("municipality_no")))
                .withColumn("county_no", F.substring("municipality_no", 1, 2)))  # unknown code -> no municipality
    return conform(good, "weather_station"), rejects


def clean_temperature(bronze):
    """bronze.frost_observations -> (temperature_station_daily, rejects)."""
    df = (bronze
          .select(F.split("sourceId", ":")[0].alias("station_id"),                       # 'SN18700:0' -> 'SN18700'
                  F.expr("to_date(convert_timezone('UTC', 'Europe/Oslo', "                   # UTC -> Norwegian time (CET/CEST)
                         "try_cast(substring(referenceTime, 1, 19) AS TIMESTAMP_NTZ)))").alias("date"),  # -> local date
                  F.explode(F.from_json("observations", OBS_SCHEMA)).alias("o"),          # one row per observation
                  "_landing_run_id")
          .select("station_id", "date", "_landing_run_id",
                  F.col("o.elementId").alias("element"), F.col("o.value").alias("value"),
                  F.col("o.timeOffset").alias("time_offset"))
          .filter(F.col("element") == "mean(air_temperature P1D)"))                       # only daily mean temperature
    good, rejects = split_rejects(df, [
        ("TEMP-01", F.col("date").isNull()),
        ("TEMP-02", F.col("value").isNull() | ~F.col("value").between(-60, 45)),
    ], "temperature_station_daily")
    good = newest_per_key(good, ["station_id", "date"],
                          [F.when(F.col("time_offset") == "PT0H", 0).otherwise(1),  # PT0H first ...
                           "time_offset",                                          # ... then the lowest offset
                           F.desc("_landing_run_id")])                             # ... newest file
    return conform(good.withColumnRenamed("value", "mean_temp_c"), "temperature_station_daily"), rejects


def temperature_per_municipality(temp_station, stations, municipalities):
    """Every municipality x day: average of own stations, else county average (Spark SQL)."""
    s = temp_station.sparkSession                               # the session these DataFrames belong to
    temp_station.createOrReplaceTempView("t_station")
    stations.createOrReplaceTempView("t_meta")
    municipalities.createOrReplaceTempView("t_muni")
    df = s.sql("""
        WITH obs AS (                                   -- temperature with the station's municipality and county
            SELECT t.date, t.mean_temp_c, m.municipality_no, m.county_no
            FROM t_station t JOIN t_meta m ON t.station_id = m.station_id
            WHERE m.municipality_no IS NOT NULL),
        own AS (SELECT municipality_no, date, AVG(mean_temp_c) AS temp, COUNT(*) AS n
                FROM obs GROUP BY municipality_no, date),
        county AS (SELECT county_no, date, AVG(mean_temp_c) AS temp, COUNT(*) AS n
                   FROM obs GROUP BY county_no, date),
        grid AS (SELECT m.municipality_no, m.county_no, d.date      -- every municipality x every day
                 FROM t_muni m CROSS JOIN (SELECT DISTINCT date FROM t_station) d)
        SELECT g.municipality_no, g.date,
               ROUND(COALESCE(o.temp, c.temp), 2)                       AS mean_temp_c,
               COALESCE(o.n, c.n)                                       AS n_stations,
               CASE WHEN o.temp IS NOT NULL THEN 'stations'
                    WHEN c.temp IS NOT NULL THEN 'county_average' END   AS method
        FROM grid g
        LEFT JOIN own o    ON o.municipality_no = g.municipality_no AND o.date = g.date
        LEFT JOIN county c ON c.county_no = g.county_no AND c.date = g.date
        WHERE COALESCE(o.temp, c.temp) IS NOT NULL                       -- no data at all -> no row
    """)
    return conform(df, "temperature_municipality_daily")

### 4.4 Norgespris consumption per price area (Elhub)
Typed and de-duplicated, same rules as consumption. The kWh columns: with Norgespris, without Norgespris, and total (`NORGESPRIS_CONSUMPTION`).
Elhub writes counts as `10702.0000000000`, so they go through `DECIMAL` before `INT`. Optional dataset: skipped if it is not in Bronze.

In [10]:
def clean_norgespris(bronze):
    """bronze.norgespris_consumption_per_group_eac_mba -> (norgespris_price_area_daily, rejects)."""
    def kwh(c):                                                 # text -> DECIMAL, '' -> NULL
        return F.expr(f"try_cast(nullif(trim(`{c}`), '') AS DECIMAL(18,3))")

    def count(c):                                               # '10702.0000000000' -> 10702
        return kwh(c).cast("int")

    df = bronze.select(
        F.expr("try_cast(substring(START_TIME, 1, 10) AS DATE)").alias("date"),   # local date
        F.trim("PRICE_AREA").alias("price_area"),
        F.lower(F.trim("CONSUMPTION_GROUP")).alias("consumption_group"),
        kwh("ESTIMATED_ANNUAL_CONSUMPTION_FROM").alias("eac_from_kwh"),
        kwh("ESTIMATED_ANNUAL_CONSUMPTION_TO").alias("eac_to_kwh"),
        kwh("QUANTITY_KWH_NORWAY_PRICE").alias("kwh_norgespris"),
        kwh("QUANTITY_KWH_NOT_NORWAY_PRICE").alias("kwh_not_norgespris"),
        kwh("NORGESPRIS_CONSUMPTION").alias("kwh_total"),
        count("METERING_POINT_COUNT_NORWAY_PRICE").alias("metering_points_norgespris"),
        count("METERING_POINT_COUNT_NOT_NORWAY_PRICE").alias("metering_points_not_norgespris"),
        count("METERING_POINT_COUNT").alias("metering_points_total"),
        try_cast("LAST_UPDATE_TIME", "timestamp").alias("last_update_time"),
        "_landing_run_id")
    good, rejects = split_rejects(df, [
        ("NP-01", F.col("date").isNull()),
        ("NP-02", F.col("price_area").isNull() | (F.col("price_area") == "")),
        ("NP-03", (F.col("kwh_norgespris") < 0) | (F.col("kwh_not_norgespris") < 0) | (F.col("kwh_total") < 0)),
    ], "norgespris_price_area_daily")
    good = newest_per_key(good, KEYS["norgespris_price_area_daily"],
                          [F.desc_nulls_last("last_update_time"), F.desc("_landing_run_id")])
    return conform(good, "norgespris_price_area_daily"), rejects

### 4.5 Norgespris meters per municipality (Elhub)
One row per day, municipality and consumption group: how many metering points have **Norgespris** (`COUNT_NORWAYPR…`) and how many there are **in total** (`COUNT`).
* **Column names:** we find the columns by how their name *starts* (`find_col`), e.g. `MUNICIPALITY_NUM…` – robust if Elhub writes
  `MUNICIPALITY_NUMBER` or `MUNICIPALITY_NUM`, `COUNT_NORWAYPRICE` or `COUNT_NORWAY_PRICE`. `COUNT` is matched exactly (otherwise it would also match `COUNT_NORWAY…`).
* **Groups:** `Household` → `household`, `Cabins and holiday properties` → `cabin` (the names used in the consumption data); other groups just in lower case.
* **Municipality:** padded to 4 characters and matched like consumption (code, or unique name for old codes); unknown codes are kept and marked.
* **`norgespris_share`** = with Norgespris ÷ total (NULL when the total is 0).

In [11]:
GROUP_NAMES = {"cabins and holiday properties": "cabin"}         # Elhub's long names -> names used in consumption_daily


def find_col(df, prefix, exact=False):
    """Name of the column that starts with (or, with exact=True, equals) prefix, ignoring upper/lower case."""
    for c in df.columns:
        if (c.upper() == prefix) if exact else c.upper().startswith(prefix):
            return c
    raise ValueError(f"No column {'named' if exact else 'starting with'} {prefix} in {df.columns}")


def standard_group(column):
    """Lower case, and Elhub's long group names mapped to the short ones (see GROUP_NAMES)."""
    g = F.lower(F.trim(column))
    for long_name, short_name in GROUP_NAMES.items():
        g = F.when(g == long_name, F.lit(short_name)).otherwise(g)
    return g


def clean_norgespris_count(bronze, municipalities):
    """bronze.norgespris_count_per_municipality_consumption_group -> (norgespris_count_daily, rejects)."""
    def count(c):                                               # '569' or '569.0000000000' -> 569
        return F.expr(f"try_cast(nullif(trim(`{c}`), '') AS DECIMAL(18,3))").cast("int")

    df = bronze.select(
        F.expr(f"try_cast(substring(`{find_col(bronze, 'DATE', exact=True)}`, 1, 10) AS DATE)").alias("date"),
        F.lpad(F.trim(F.col(f"`{find_col(bronze, 'MUNICIPALITY_NUM')}`")), 4, "0").alias("municipality_code"),
        F.col(f"`{find_col(bronze, 'MUNICIPALITY', exact=True)}`").alias("name"),
        standard_group(F.col(f"`{find_col(bronze, 'CONSUMPTION_GR')}`")).alias("consumption_group"),
        count(find_col(bronze, "COUNT_NORWAY")).alias("metering_points_norgespris"),
        count(find_col(bronze, "COUNT", exact=True)).alias("metering_points_total"),
        "_landing_run_id")
    good, rejects = split_rejects(df, [
        ("NPC-01", F.col("date").isNull()),
        ("NPC-02", F.col("municipality_code").isNull() | (F.col("municipality_code") == "")),
        ("NPC-03", F.col("metering_points_norgespris").isNull() | F.col("metering_points_total").isNull()
                   | (F.col("metering_points_norgespris") < 0)
                   | (F.col("metering_points_norgespris") > F.col("metering_points_total"))),
    ], "norgespris_count_daily")
    good = match_municipality(good, "municipality_code", "name", municipalities)   # old codes -> today's code
    good = newest_per_key(good, KEYS["norgespris_count_daily"], [F.desc("_landing_run_id")])  # same day twice -> newest
    good = (good.withColumn("known_municipality", F.col("municipality_match").isNotNull())
                .withColumn("norgespris_share",
                            F.when(F.col("metering_points_total") > 0,
                                   F.round(F.col("metering_points_norgespris") / F.col("metering_points_total"), 4))))
    return conform(good, "norgespris_count_daily"), rejects

---
# 5. Run Silver and measure quality

### 5.1 `run_silver()` – the whole Silver stage
For every table: read Bronze → clean → write `silver.<table>` → collect metrics. At the end: all rejects to `silver.rejects` (replaced each run,
with the run id) and the metrics **appended** to `silver.quality_metrics`.
Required inputs: population, consumption, Frost stations and observations. The two Norgespris datasets are optional.

Metric names (per table): `bronze_rows`, `rejected_rows`, `duplicates_removed`, `silver_rows`, `duplicate_keys` (must be 0), plus a few
table-specific ones (kWh in unknown municipalities, incomplete days, municipalities with own stations / county average).

In [12]:
def duplicate_keys(df, table):
    """Number of key values that occur more than once (must be 0 - proves the grain)."""
    return df.groupBy(*KEYS[table]).count().filter("count > 1").count()


def run_silver():
    """Build all Silver tables from the bronze schema; returns a summary dict (same contract as the other stages)."""
    global spark
    start_time = time.time()
    if not PG_PASSWORD or PG_PASSWORD.startswith("your-"):
        logger.error("NORGESPRIS_PG_PASSWORD is not set (in .env or as GitHub secret)")
        return {"status": "failed", "elapsed_time": 0.0}
    spark = get_spark()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    available = bronze_tables()
    required = ["population_07459", "consumption_per_group_municipality_hour", "frost_stations", "frost_observations"]
    missing = [t for t in required if t not in available]
    if missing:                                                 # Silver cannot be built without these
        logger.error(f"Missing Bronze tables {missing} - run the bronze and postgres stages first")
        return {"status": "failed", "elapsed_time": time.time() - start_time}
    first_day, last_day = silver_period()                       # the period from .env (same as the Frost download)
    logger.info(f"Silver run {run_id} (env={ENV}, period {first_day} .. {last_day}) -> {PG_URL}")
    metrics, rejects, rows = [], [], {}

    def metric(table, name, value):                             # collect one quality number
        metrics.append((run_id, table, name, float(value or 0)))

    # --- 4.1 municipalities and population ----------------------------------------------------
    pop_in = read_bronze("population_07459")
    population, municipality, rej = clean_population(pop_in)
    population = population.filter(F.col("year").between(first_day.year, last_day.year))  # years of the period
    rejects.append(rej)
    municipality = municipality.persist()                       # small, used by several steps below
    for table, df in (("population_yearly", population), ("municipality", municipality)):
        rows[table] = write_silver(df, table)
        metric(table, "silver_rows", rows[table])
        metric(table, "duplicate_keys", duplicate_keys(df, table))
    no_name = municipality.filter(F.col("municipality_name").isNull() | (F.col("municipality_name") == "")).count()
    metric("municipality", "municipalities_without_name", no_name)  # must be 0
    if no_name:                                                 # SSB delivered codes only -> check SSB_URL in Bronze
        logger.warning(f"  {no_name} municipalities without a name - does the SSB request ask for "
                       f"'usecodesandtexts' (SSB_URL in bronze_pipeline.py)?")
    metric("population_yearly", "bronze_rows", pop_in.count())
    metric("population_yearly", "rejected_rows", rej.count())

    # --- 4.2 consumption --------------------------------------------------------------------------
    cons_in = read_bronze("consumption_per_group_municipality_hour", split_on="START_TIME")
    cons_in = cons_in.persist(StorageLevel.MEMORY_AND_DISK)     # read PostgreSQL once, use it several times
    consumption, rej, hourly = clean_consumption(cons_in, municipality)
    consumption = in_period(consumption, first_day, last_day)   # only the days of the period
    rejects.append(rej)
    n_in, n_rej = cons_in.count(), rej.count()
    rows["consumption_daily"] = write_silver(consumption, "consumption_daily")
    metric("consumption_daily", "bronze_rows", n_in)
    metric("consumption_daily", "rejected_rows", n_rej)
    metric("consumption_daily", "duplicates_removed", n_in - n_rej - hourly.count())
    metric("consumption_daily", "silver_rows", rows["consumption_daily"])
    metric("consumption_daily", "duplicate_keys", duplicate_keys(consumption, "consumption_daily"))
    totals = consumption.agg(F.sum("kwh").alias("all"),
                             F.sum(F.when(~F.col("known_municipality"), F.col("kwh"))).alias("unknown"),
                             F.sum(F.when(~F.col("complete_day"), 1)).alias("incomplete")).first()
    metric("consumption_daily", "kwh_total", totals["all"])
    days = consumption.agg(F.min("date").alias("first"), F.max("date").alias("last")).first()
    metric("consumption_daily", "first_day_yyyymmdd", int(days["first"].strftime("%Y%m%d")) if days["first"] else 0)
    metric("consumption_daily", "last_day_yyyymmdd", int(days["last"].strftime("%Y%m%d")) if days["last"] else 0)
    metric("consumption_daily", "kwh_unknown_municipality", totals["unknown"])
    metric("consumption_daily", "incomplete_days", totals["incomplete"])
    cons_in.unpersist()

    # --- 4.3 weather -----------------------------------------------------------------------------
    st_in = read_bronze("frost_stations")
    stations, rej = clean_stations(st_in, municipality)
    rejects.append(rej)
    stations = stations.persist()
    rows["weather_station"] = write_silver(stations, "weather_station")
    for name, cond in (("stations_with_municipality", "municipality_no IS NOT NULL"),
                       ("stations_matched_by_name", "municipality_match = 'name'")):
        metric("weather_station", name, stations.filter(cond).count())
    metric("weather_station", "silver_rows", rows["weather_station"])

    obs_in = read_bronze("frost_observations", split_on="referenceTime").persist(StorageLevel.MEMORY_AND_DISK)
    temp_station, rej = clean_temperature(obs_in)
    temp_station = in_period(temp_station, first_day, last_day)  # (Frost already downloads only this period)
    rejects.append(rej)
    temp_station = temp_station.persist()
    rows["temperature_station_daily"] = write_silver(temp_station, "temperature_station_daily")
    metric("temperature_station_daily", "bronze_rows", obs_in.count())
    metric("temperature_station_daily", "rejected_rows", rej.count())
    metric("temperature_station_daily", "silver_rows", rows["temperature_station_daily"])
    metric("temperature_station_daily", "duplicate_keys", duplicate_keys(temp_station, "temperature_station_daily"))

    temp_muni = temperature_per_municipality(temp_station, stations, municipality)
    rows["temperature_municipality_daily"] = write_silver(temp_muni, "temperature_municipality_daily")
    for method in ("stations", "county_average"):              # how many municipalities use which method
        n = temp_muni.filter(F.col("method") == method).select("municipality_no").distinct().count()
        metric("temperature_municipality_daily", f"municipalities_{method}", n)
    metric("temperature_municipality_daily", "municipalities_without_temperature",
           municipality.join(temp_muni.select("municipality_no").distinct(), "municipality_no", "left_anti").count())
    metric("temperature_municipality_daily", "silver_rows", rows["temperature_municipality_daily"])
    obs_in.unpersist(); temp_station.unpersist(); stations.unpersist()

    # --- 4.4 Norgespris (optional) -----------------------------------------------------------------
    if "norgespris_consumption_per_group_eac_mba" in available:
        np_in = read_bronze("norgespris_consumption_per_group_eac_mba")
        norgespris, rej = clean_norgespris(np_in)
        norgespris = in_period(norgespris, first_day, last_day)
        rejects.append(rej)
        rows["norgespris_price_area_daily"] = write_silver(norgespris, "norgespris_price_area_daily")
        metric("norgespris_price_area_daily", "bronze_rows", np_in.count())
        metric("norgespris_price_area_daily", "rejected_rows", rej.count())
        metric("norgespris_price_area_daily", "silver_rows", rows["norgespris_price_area_daily"])
    else:
        logger.info("  norgespris_consumption_per_group_eac_mba not in Bronze - skipped (optional)")

    # --- 4.5 Norgespris meters per municipality (optional) ---------------------------------------------
    if "norgespris_count_per_municipality_consumption_group" in available:
        npc_in = read_bronze("norgespris_count_per_municipality_consumption_group", split_on="DATE")
        npc_in = npc_in.persist(StorageLevel.MEMORY_AND_DISK)
        norgespris_count, rej = clean_norgespris_count(npc_in, municipality)
        norgespris_count = in_period(norgespris_count, first_day, last_day)
        rejects.append(rej)
        rows["norgespris_count_daily"] = write_silver(norgespris_count, "norgespris_count_daily")
        metric("norgespris_count_daily", "bronze_rows", npc_in.count())
        metric("norgespris_count_daily", "rejected_rows", rej.count())
        metric("norgespris_count_daily", "silver_rows", rows["norgespris_count_daily"])
        metric("norgespris_count_daily", "duplicate_keys", duplicate_keys(norgespris_count, "norgespris_count_daily"))
        metric("norgespris_count_daily", "rows_unknown_municipality",
               norgespris_count.filter(~F.col("known_municipality")).count())
        npc_in.unpersist()
    else:
        logger.info("  norgespris_count_per_municipality_consumption_group not in Bronze - skipped (optional)")
    municipality.unpersist()

    # --- rejects and metrics -------------------------------------------------------------------------
    all_rejects = rejects[0]
    for r in rejects[1:]:
        all_rejects = all_rejects.unionByName(r)                 # same three columns in every rejects DataFrame
    all_rejects = all_rejects.select(F.lit(run_id).alias("run_id"), "*")
    rows["rejects"] = write_silver(all_rejects, "rejects")
    measured_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    metrics_df = spark.createDataFrame([(r, measured_at, t, m, v) for r, t, m, v in metrics],
                                       "run_id string, measured_at string, table_name string, metric string, value double")
    metrics_df.write.jdbc(PG_URL, "silver.quality_metrics", mode="append", properties=PG_PROPERTIES)  # history

    elapsed = time.time() - start_time
    failed_keys = [m for m in metrics if m[2] == "duplicate_keys" and m[3] > 0]  # grain broken -> stage fails
    status = "failed" if failed_keys else "success"
    logger.info(f"Silver done in {elapsed:.1f}s: {rows}" + (f" DUPLICATE KEYS: {failed_keys}" if failed_keys else ""))
    return {"status": status, "elapsed_time": elapsed, "run_id": run_id, "rows": rows}

In [13]:
silver_result = run_silver()                                    # build all Silver tables (a few seconds in dev)
silver_result

2026-11-13 01:50:22,467 - silver - INFO - Silver run 20261113T005022369137Z (env=dev, period 2024-01-01 .. 2026-11-30) -> jdbc:postgresql://localhost:5432/norgespris_dev
2026-11-13 01:50:24,252 - silver - INFO -   silver.population_yearly: 1,071 rows
2026-11-13 01:50:25,174 - silver - INFO -   silver.municipality: 357 rows
2026-11-13 01:54:39,212 - silver - INFO -   silver.consumption_daily: 1,123,404 rows
2026-11-13 01:57:03,200 - silver - INFO -   silver.weather_station: 2,307 rows  
2026-11-13 01:57:34,539 - silver - INFO -   silver.temperature_station_daily: 913,946 rows
2026-11-13 01:57:40,438 - silver - INFO -   silver.temperature_municipality_daily: 373,779 rows
2026-11-13 01:57:47,708 - silver - INFO -   silver.norgespris_price_area_daily: 41,668 rows
2026-11-13 01:57:51,663 - silver - INFO -   silver.norgespris_count_daily: 289,645 rows
2026-11-13 01:59:54,405 - silver - INFO -   silver.rejects: 94 rows             
2026-11-13 01:59:55,831 - silver - INFO - Silver done in 573.

{'status': 'success',
 'elapsed_time': 573.4761130809784,
 'run_id': '20261113T005022369137Z',
 'rows': {'population_yearly': 1071,
  'municipality': 357,
  'consumption_daily': 1123404,
  'weather_station': 2307,
  'temperature_station_daily': 913946,
  'temperature_municipality_daily': 373779,
  'norgespris_price_area_daily': 41668,
  'norgespris_count_daily': 289645,
  'rejects': 94}}

### 5.2 Quality evidence – this run compared with the previous run
`silver.quality_metrics` keeps every run. The table below puts the newest run next to the previous one – the guide's "metrics before and after".
`duplicate_keys` must be 0 for every table (otherwise the stage fails); `rejected_rows` should be small and explainable from `silver.rejects`.

In [14]:
q = pg_query("SELECT * FROM silver.quality_metrics")            # all runs
runs = [r["run_id"] for r in q.select("run_id").distinct().orderBy(F.desc("run_id")).limit(2).collect()]
latest = q.filter(F.col("run_id") == runs[0]).select("table_name", "metric", F.col("value").alias("this_run"))
if len(runs) > 1:                                               # compare with the previous run
    previous = q.filter(F.col("run_id") == runs[1]).select("table_name", "metric", F.col("value").alias("previous_run"))
    latest = latest.join(previous, ["table_name", "metric"], "left")
latest.orderBy("table_name", "metric").show(60, truncate=False)

+------------------------------+----------------------------------+-------------------+-------------------+
|table_name                    |metric                            |this_run           |previous_run       |
+------------------------------+----------------------------------+-------------------+-------------------+
|consumption_daily             |bronze_rows                       |2.8250496E7        |2.8250496E7        |
|consumption_daily             |duplicate_keys                    |0.0                |0.0                |
|consumption_daily             |duplicates_removed                |0.0                |0.0                |
|consumption_daily             |first_day_yyyymmdd                |2.0240101E7        |NULL               |
|consumption_daily             |incomplete_days                   |4.0                |4.0                |
|consumption_daily             |kwh_total                         |3.77151908700545E11|3.99173947107352E11|
|consumption_daily          

In [15]:
pg_query("""SELECT dataset, rule_id, COUNT(*) AS n, MIN(record) AS example
            FROM silver.rejects GROUP BY dataset, rule_id ORDER BY dataset, rule_id""").show(truncate=80)

+---------------------------+-------+---+--------------------------------------------------------------------------------+
|                    dataset|rule_id|  n|                                                                         example|
+---------------------------+-------+---+--------------------------------------------------------------------------------+
|          consumption_daily|CONS-03| 37|{"start_time_text":"2024-01-01T09:00:00.000+01:00","start_time":"2024-01-01T0...|
|norgespris_price_area_daily|  NP-03| 44|{"date":"2024-05-16","price_area":"NO3","consumption_group":"cabin","kwh_norg...|
|          population_yearly| SSB-01| 12|{"region":"K-21-22 - K-21-22 Svalbard og Jan Mayen","_landing_run_id":"202611...|
|          population_yearly| SSB-02|  1|{"region":"K-1580 - K-1580 Haram","_landing_run_id":"20261112T232839993654Z",...|
+---------------------------+-------+---+--------------------------------------------------------------------------------+



### 5.3 Performance evidence
Instead of tuning by assumption we look at what Spark does:
* the number of partitions of the big JDBC read (one per month + one catch-all),
* the plan of the municipality join – `BroadcastHashJoin` means the small table is sent to every task and the big one is **not** shuffled.

In [16]:
cons = read_bronze("consumption_per_group_municipality_hour", split_on="START_TIME")
print("JDBC partitions (months + 1):", cons.rdd.getNumPartitions())
plain = spark.read.jdbc(PG_URL, "bronze.consumption_per_group_municipality_hour", properties=JDBC_READ)
print("Without predicates:", plain.rdd.getNumPartitions(), "partition")

muni = pg_query("SELECT * FROM silver.municipality")
daily, _, _ = clean_consumption(cons, muni)
daily.explain(mode="formatted")                                 # look for BroadcastHashJoin and the Exchange (shuffle) steps

JDBC partitions (months + 1): 38
Without predicates: 1 partition
== Physical Plan ==
AdaptiveSparkPlan (33)
+- Project (32)
   +- HashAggregate (31)
      +- Exchange (30)
         +- HashAggregate (29)
            +- Project (28)
               +- Filter (27)
                  +- Window (26)
                     +- WindowGroupLimit (25)
                        +- Sort (24)
                           +- Exchange (23)
                              +- WindowGroupLimit (22)
                                 +- Sort (21)
                                    +- Project (20)
                                       +- BroadcastHashJoin LeftOuter BuildRight (19)
                                          :- BroadcastHashJoin LeftOuter BuildRight (7)
                                          :  :- Project (3)
                                          :  :  +- Filter (2)
                                          :  :     +- Scan JDBCRelation(bronze.consumption_per_group_municipality_hour) [numPartit

### 5.4 A look at the result
Consumption per day, joined with temperature, population and the Norgespris share – the shape Gold will build on
(the join on `consumption_group` works because both tables use `household` / `cabin`).

In [ ]:
pg_query("""
    SELECT c.municipality_no, m.municipality_name, c.date, c.consumption_group, c.kwh, c.n_hours,
           t.mean_temp_c, t.method, p.population, n.norgespris_share
    FROM silver.consumption_daily c
    JOIN silver.municipality m USING (municipality_no)
    LEFT JOIN silver.temperature_municipality_daily t USING (municipality_no, date)
    LEFT JOIN silver.norgespris_count_daily n USING (municipality_no, date, consumption_group)
    LEFT JOIN silver.population_yearly p ON p.municipality_no = c.municipality_no AND p.year = EXTRACT(YEAR FROM c.date)
    ORDER BY c.municipality_no, c.date, c.consumption_group
    LIMIT 12""").show(truncate=False)

2026-11-14 11:05:03,332 - py4j.clientserver - INFO - Error while sending or receiving.
Traceback (most recent call last):
  File "/Users/x/UiS/DAT535 Data Engineering/spark-env/lib/python3.14/site-packages/py4j/clientserver.py", line 527, in send_command
    self.socket.sendall(command.encode("utf-8"))
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
BrokenPipeError: [Errno 32] Broken pipe
2026-11-14 11:05:03,335 - py4j.clientserver - INFO - Closing down clientserver connection
2026-11-14 11:05:03,336 - root - INFO - Exception while sending command.
Traceback (most recent call last):
  File "/Users/x/UiS/DAT535 Data Engineering/spark-env/lib/python3.14/site-packages/py4j/clientserver.py", line 527, in send_command
    self.socket.sendall(command.encode("utf-8"))
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
BrokenPipeError: [Errno 32] Broken pipe

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/x/UiS/DAT535 Data 

ConnectionRefusedError: [Errno 61] Connection refused

2026-11-14 11:05:03,922 - py4j.clientserver - INFO - Error while sending or receiving.
Traceback (most recent call last):
  File "/Users/x/UiS/DAT535 Data Engineering/spark-env/lib/python3.14/site-packages/py4j/clientserver.py", line 527, in send_command
    self.socket.sendall(command.encode("utf-8"))
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
BrokenPipeError: [Errno 32] Broken pipe
2026-11-14 11:05:03,923 - py4j.clientserver - INFO - Closing down clientserver connection
2026-11-14 11:05:03,924 - root - INFO - Exception while sending command.
Traceback (most recent call last):
  File "/Users/x/UiS/DAT535 Data Engineering/spark-env/lib/python3.14/site-packages/py4j/clientserver.py", line 527, in send_command
    self.socket.sendall(command.encode("utf-8"))
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^
BrokenPipeError: [Errno 32] Broken pipe

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/x/UiS/DAT535 Data 

---
# 6. Recommendations from notebook 01 – what Silver does with them, and what is still open

| # | Recommendation | Status in Silver |
|---|---|---|
| 1 | Elhub API serves only ~3 rolling years | ⚠️ **Open:** Silver is rebuilt from the newest Bronze (overwrite), so the oldest month disappears when Elhub drops it. Fine for this semester; later improvement: keep old days in Silver (append missing dates instead of overwrite). |
| 2 | Municipalities without a weather station | ✅ county average (`method = 'county_average'`); metric `municipalities_without_temperature` |
| 3 | Several daily-mean series per station | ✅ one per station and day: `PT0H` first (`clean_temperature`) |
| 4 | Norgespris count per municipality | ✅ `silver.norgespris_count_daily` (4.5): meters with Norgespris, total and `norgespris_share` per municipality, day and group – no own distribution key needed |
| 5 | Municipality `0` (unknown) | ✅ kept, `known_municipality = false`; metric `kwh_unknown_municipality` |
| 6 | Different municipality codes / 2024 renumbering | ✅ 4-digit codes; old codes matched by unique name (`match_municipality`); metric `stations_matched_by_name` |
| 7 | Time zones | ✅ Elhub: local date. ✅ Frost: `referenceTime` converted from UTC to `Europe/Oslo` (CET/CEST) before taking the date. Still write in the report that a `PT0H` daily mean covers 00–24 UTC. |
| 8 | Broken Norwegian letters | ✅ names from SSB only; joins on numbers |
| 9 | ZIP / gzip | ✅ handled in landing (notebook 01) |
| 10 | Different separators | ✅ handled in Bronze (notebook 01) |
| 11 | SSB is wide | ✅ unpivoted, non-municipality rows rejected (`SSB-01`) |
| 12 | Revised / overlapping data | ✅ newest version per key |
| 13–15 | Small text corrections in the definition | ⚠️ **Open:** update the project definition document (weather *per station* via Frost; secrets; what happens to Gold when Silver fails) |

### New findings while building Silver
1. **Spark 4 runs in ANSI mode** – `cast('abc' AS DECIMAL)` stops the whole job instead of returning NULL. All conversions from Bronze text use `try_cast`; keep it that way in Gold.
2. **JDBC reads without `fetchsize`** load the whole result into memory, and without `predicates` everything comes through one connection. Fixed in `read_bronze`; use the same helper in Gold.
3. **Frost municipality codes may be pre-2024** (e.g. Drammen `3005`, today `3301`). Matched by name; names that exist twice in Norway (e.g. *Herøy*, *Våler*) cannot be matched that way – check `stations_with_municipality` and the stations with empty `municipality_no`.
4. **The Norgespris count file uses other group names** (`Household`, `Cabins and holiday properties`) than the consumption file (`household`, `cabin`) – standardised in 4.5; if Elhub adds more groups, add them to `GROUP_NAMES`.
5. **The Norgespris price-area file has rows where the consumption band is empty and the with/without columns are 0, but the total is not** – the meaning of those rows must be checked with Elhub's documentation before Gold uses them.
6. ✅ **Workflows (fixed):** the unit tests now run once, *before* the pipeline, in the dev and prod workflows, so broken code never writes data. The README now uses the branch name the dev workflow listens to: `dev_branch`.
7. ✅ **Municipality names were empty (fixed):** SSB's API writes only codes in CSV by default. `SSB_URL` (notebook 01 / `bronze_pipeline.py`) now asks for `usecodesandtexts`; Silver checks it with the metric `municipalities_without_name`.
8. **Silver is a full rebuild** every run. That is simple and idempotent; at ~0.5 million consumption rows per year it is fast enough. Revisit only if a run gets slow.

---
# 7. How to run

| Where | Command |
|---|---|
| Notebook | run notebook 01 (or the `bronze` + `postgres` stages) first, then all cells here |
| Other period (e.g. earlier months) | change `NORGESPRIS_<ENV>_START_MONTH` in `.env` (GitHub: repository variable), then `python run_pipeline.py all` – `bronze` downloads the missing Frost months, `postgres` reloads, `silver` rebuilds |
| Terminal | `python run_pipeline.py silver` (only Silver) or `python run_pipeline.py all` (bronze → postgres → silver) |
| Tests | `pytest -q tests/` |
| GitHub Actions | the workflows run `python run_pipeline.py all`; the prod workflow can run `silver` alone |

Check the result in `psql`: `\dt silver.*`, `SELECT * FROM silver.quality_metrics ORDER BY measured_at DESC LIMIT 20;`

**Next notebook (03):** Gold – the daily consumption per inhabitant before/after Norgespris, adjusted for temperature, with Spark SQL.